# silver_policies

Largest and messiest silver table in the project. 10,000 rows from two bronze sources, ~2.5% intentionally seeded dirty data, and FK validation against both silver_carriers and silver_agents. This is where the silver layer earns its keep.

The pattern is identical to silver_agents (UNION, source priority, reject quarantine) but the validation is heavier and the stakes are higher. Every downstream table, silver_claims, silver_commission_payouts, and all four gold facts — depends on silver_policies being clean.

## Dirty data to catch

Five corruption types were seeded in Phase 1:

1. Null premium (1% of rows)
2. Negative premium (0.5% of rows)
3. Malformed effective_date stored as "2025-13-45" (0.5% of rows)
4. Null carrier_id (0.5% of rows)
5. agent_id = "AGT-99999" (FK to nonexistent agent, 0.5% of rows)

All five are caught here and quarantined in silver_policies_rejects. None make it into silver_policies.

## Foreign Key validation

Two FK checks run against silver (not bronze) reference tables:

- carrier_id must exist in silver_carriers
- agent_id must exist in silver_agents

We validate against silver, not bronze, because silver is the layer that has been cleaned and validated. A carrier that failed silver_carriers validation wouldn't be in silver_carriers, so it shouldn't be a valid FK target either.

## Source and output

- `bronze_citadel_mga.bronze_policies` (CSV, 10,000 rows)
- `bronze_citadel_mga.bronze_policies_api` (API, 10,000 rows)

Outputs:
- `silver_citadel_mga.silver_policies` (clean rows)
- `silver_citadel_mga.silver_policies_rejects` (dirty rows with reject_reason)

## Source priority

Same as silver_agents. CSV wins on conflicts. API-only rows are included.


In [1]:
from pyspark.sql.functions import lit, current_timestamp, col, when, concat_ws, to_date
from pyspark.sql.types import StringType
import uuid
from datetime import datetime

SILVER_BATCH_ID = f"silver_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# Two bronze sources.
SOURCE_CSV = "bronze_citadel_mga.dbo.bronze_policies"
SOURCE_API = "bronze_citadel_mga.dbo.bronze_policies_api"

# Two silver outputs.
TARGET_TABLE  = "silver_policies"
REJECTS_TABLE = "silver_policies_rejects"

# FK reference tables — validate against silver, not bronze.
REF_CARRIERS = "silver_citadel_mga.dbo.silver_carriers"
REF_AGENTS   = "silver_citadel_mga.dbo.silver_agents"

# Valid policy statuses.
VALID_STATUSES = {"ACTIVE", "EXPIRED", "CANCELLED", "PENDING"}

print(f"Silver batch ID: {SILVER_BATCH_ID}")
print(f"CSV source:      {SOURCE_CSV}")
print(f"API source:      {SOURCE_API}")
print(f"Target:          {TARGET_TABLE}")
print(f"Rejects:         {REJECTS_TABLE}")
print(f"FK ref carriers: {REF_CARRIERS}")
print(f"FK ref agents:   {REF_AGENTS}")

StatementMeta(, 7ee4c904-f779-4dea-a6d9-2234662fe186, 3, Finished, Available, Finished, False)

Silver batch ID: silver_20260603_010842_2e7daf15
CSV source:      bronze_citadel_mga.dbo.bronze_policies
API source:      bronze_citadel_mga.dbo.bronze_policies_api
Target:          silver_policies
Rejects:         silver_policies_rejects
FK ref carriers: silver_citadel_mga.dbo.silver_carriers
FK ref agents:   silver_citadel_mga.dbo.silver_agents


In [2]:
# Read both bronze sources and confirm schemas match before UNION.
# Also load FK reference sets into memory — we'll use them in validation.

csv_df = spark.read.table(SOURCE_CSV)
api_df = spark.read.table(SOURCE_API)

print(f"CSV source:  {csv_df.count()} rows, {len(csv_df.columns)} cols")
print(f"API source:  {api_df.count()} rows, {len(api_df.columns)} cols")
print()
print(f"CSV columns: {csv_df.columns}")
print()
print(f"API columns: {api_df.columns}")
print()

# Load FK reference sets into Python sets for efficient lookup.
# Collecting to driver is fine for small reference tables (20 carriers,
# 200 agents). Don't do this for large tables.
valid_carrier_ids = set(
    row["carrier_id"]
    for row in spark.read.table(REF_CARRIERS).select("carrier_id").collect()
)
valid_agent_ids = set(
    row["agent_id"]
    for row in spark.read.table(REF_AGENTS).select("agent_id").collect()
)

print(f"Valid carrier_ids loaded: {len(valid_carrier_ids)}")
print(f"Valid agent_ids loaded:   {len(valid_agent_ids)}")
print()

# Check domain column mismatches between sources.
csv_domain = set(csv_df.columns) - {"_ingestion_timestamp", "_source_file", "_batch_id", "_ingestion_method"}
api_domain  = set(api_df.columns) - {"_ingestion_timestamp", "_source_endpoint", "_batch_id", "_ingestion_method"}

only_in_csv = csv_domain - api_domain
only_in_api = api_domain - csv_domain

print(f"Domain cols only in CSV: {only_in_csv if only_in_csv else 'none'}")
print(f"Domain cols only in API: {only_in_api if only_in_api else 'none'}")

StatementMeta(, 7ee4c904-f779-4dea-a6d9-2234662fe186, 4, Finished, Available, Finished, False)

CSV source:  10000 rows, 14 cols
API source:  10000 rows, 14 cols

CSV columns: ['policy_id', 'program_code', 'carrier_id', 'agent_id', 'effective_date', 'expiration_date', 'premium', 'status', 'insured_name', 'insured_state', '_ingestion_timestamp', '_source_file', '_batch_id', '_ingestion_method']

API columns: ['agent_id', 'carrier_id', 'effective_date', 'expiration_date', 'insured_name', 'insured_state', 'policy_id', 'premium', 'program_code', 'status', '_ingestion_timestamp', '_source_endpoint', '_batch_id', '_ingestion_method']

Valid carrier_ids loaded: 20
Valid agent_ids loaded:   200

Domain cols only in CSV: none
Domain cols only in API: none


In [3]:
# Standardize lineage columns before UNION so both sources have the same shape.
csv_standardized = (
    csv_df
    .withColumnRenamed("_batch_id", "_bronze_batch_id")
    .withColumn("_source_system", lit("csv"))
    .drop("_ingestion_timestamp", "_source_file", "_ingestion_method")
)

api_standardized = (
    api_df
    .withColumnRenamed("_batch_id", "_bronze_batch_id")
    .withColumn("_source_system", lit("api"))
    .drop("_ingestion_timestamp", "_source_endpoint", "_ingestion_method")
)

unioned_df = csv_standardized.unionByName(api_standardized)

print(f"CSV rows after standardize:  {csv_standardized.count()}")
print(f"API rows after standardize:  {api_standardized.count()}")
print(f"Combined rows after UNION:   {unioned_df.count()}")
print()
print(f"Columns: {unioned_df.columns}")

StatementMeta(, 7ee4c904-f779-4dea-a6d9-2234662fe186, 5, Finished, Available, Finished, False)

CSV rows after standardize:  10000
API rows after standardize:  10000
Combined rows after UNION:   20000

Columns: ['policy_id', 'program_code', 'carrier_id', 'agent_id', 'effective_date', 'expiration_date', 'premium', 'status', 'insured_name', 'insured_state', '_bronze_batch_id', '_source_system']


In [4]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

priority_df = unioned_df.withColumn(
    "_priority",
    when(col("_source_system") == "csv", 1).otherwise(2)
)

window_spec = Window.partitionBy("policy_id").orderBy("_priority")

ranked_df = priority_df.withColumn("_rank", row_number().over(window_spec))

deduped_df = (
    ranked_df
    .filter(col("_rank") == 1)
    .drop("_priority", "_rank")
)

csv_count     = deduped_df.filter(col("_source_system") == "csv").count()
api_only_count = deduped_df.filter(col("_source_system") == "api").count()

print(f"Rows after deduplication:  {deduped_df.count()}")
print(f"  From CSV:                {csv_count}")
print(f"  From API only:           {api_only_count}")
print()
print(f"Columns: {deduped_df.columns}")

StatementMeta(, 7ee4c904-f779-4dea-a6d9-2234662fe186, 6, Finished, Available, Finished, False)

Rows after deduplication:  10000
  From CSV:                10000
  From API only:           0

Columns: ['policy_id', 'program_code', 'carrier_id', 'agent_id', 'effective_date', 'expiration_date', 'premium', 'status', 'insured_name', 'insured_state', '_bronze_batch_id', '_source_system']


In [6]:
# Validate deduped rows against business rules and FK constraints.
# Five seeded corruption types to catch, plus a status sanity check.
# Failing rows go to rejects with a reject_reason explaining why.

valid_carrier_list = list(valid_carrier_ids)
valid_agent_list   = list(valid_agent_ids)

reject_reasons = (
    deduped_df
    # Rule 1: premium must not be null
    .withColumn(
        "r_null_premium",
        when(col("premium").isNull(), "null premium").otherwise(None)
    )
    # Rule 2: premium must be positive
    .withColumn(
        "r_negative_premium",
        when(
            col("premium").isNotNull() & (col("premium") < 0),
            "negative premium"
        ).otherwise(None)
    )
    # Rule 3: effective_date must parse as a real date.
    # to_date() returns null for malformed strings like "2025-13-45".
    .withColumn(
        "r_bad_date",
        when(
            col("effective_date").cast("date").isNull(),
            "malformed effective_date"
        ).otherwise(None)
    )
    # Rule 4: carrier_id must not be null and must exist in silver_carriers.
    .withColumn(
        "r_bad_carrier",
        when(
            col("carrier_id").isNull() | ~col("carrier_id").isin(valid_carrier_list),
            "invalid carrier_id"
        ).otherwise(None)
    )
    # Rule 5: agent_id must not be null and must exist in silver_agents.
    .withColumn(
        "r_bad_agent",
        when(
            col("agent_id").isNull() | ~col("agent_id").isin(valid_agent_list),
            "invalid agent_id"
        ).otherwise(None)
    )
    # Rule 6: status must be a known value.
    .withColumn(
        "r_bad_status",
        when(
            ~col("status").isin(list(VALID_STATUSES)),
            "invalid status"
        ).otherwise(None)
    )
    # Concatenate all failure reasons into one string.
    # concat_ws skips nulls automatically, so clean rows get an empty string.
    .withColumn(
        "reject_reason",
        concat_ws(" | ",
            col("r_null_premium"), col("r_negative_premium"),
            col("r_bad_date"), col("r_bad_carrier"),
            col("r_bad_agent"), col("r_bad_status")
        )
    )
    .drop("r_null_premium", "r_negative_premium", "r_bad_date",
          "r_bad_carrier", "r_bad_agent", "r_bad_status")
)

# Split into clean and rejected.
rejects_df = reject_reasons.filter(col("reject_reason") != "")
clean_df   = reject_reasons.filter(col("reject_reason") == "")

reject_count = rejects_df.count()
clean_count  = clean_df.count()
total_count  = deduped_df.count()

print(f"Total rows:    {total_count}")
print(f"Clean rows:    {clean_count}")
print(f"Rejected rows: {reject_count}")
print(f"Reject rate:   {reject_count/total_count*100:.1f}%")
print()

assert clean_count + reject_count == total_count, (
    f"Row count mismatch: clean={clean_count} + rejects={reject_count} != total={total_count}"
)
print("Row count assertion passed.")
print()

# Show reject breakdown by reason.
print("Reject breakdown:")
rejects_df.groupBy("reject_reason").count().orderBy("count", ascending=False).show(truncate=False)

StatementMeta(, 7ee4c904-f779-4dea-a6d9-2234662fe186, 8, Finished, Available, Finished, False)

Total rows:    10000
Clean rows:    9709
Rejected rows: 291
Reject rate:   2.9%

Row count assertion passed.

Reject breakdown:
+------------------------+-----+
|reject_reason           |count|
+------------------------+-----+
|null premium            |80   |
|negative premium        |62   |
|invalid agent_id        |57   |
|malformed effective_date|49   |
|invalid carrier_id      |43   |
+------------------------+-----+



In [7]:
# Add silver lineage to clean rows and write both tables.
silver_df = (
    clean_df
    .drop("reject_reason")
    .withColumn("_silver_load_timestamp", current_timestamp())
    .withColumn("_silver_batch_id", lit(SILVER_BATCH_ID))
)

rejects_silver_df = (
    rejects_df
    .withColumn("_silver_load_timestamp", current_timestamp())
    .withColumn("_silver_batch_id", lit(SILVER_BATCH_ID))
)

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)
print(f"Wrote {silver_df.count()} rows to {TARGET_TABLE}")

(
    rejects_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(REJECTS_TABLE)
)
print(f"Wrote {rejects_silver_df.count()} rows to {REJECTS_TABLE}")

StatementMeta(, 7ee4c904-f779-4dea-a6d9-2234662fe186, 9, Finished, Available, Finished, False)

Wrote 9709 rows to silver_policies
Wrote 291 rows to silver_policies_rejects


In [9]:
verify_clean   = spark.read.table(TARGET_TABLE)
verify_rejects = spark.read.table(REJECTS_TABLE)

clean_count   = verify_clean.count()
reject_count  = verify_rejects.count()
total_deduped = deduped_df.count()

print(f"silver_policies verified:")
print(f"  Clean rows:    {clean_count}")
print(f"  Rejected rows: {reject_count}")
print(f"  Total deduped: {total_deduped}")
print(f"  Math checks:   {clean_count + reject_count} == {total_deduped}: {clean_count + reject_count == total_deduped}")
print()

required_lineage = ["_bronze_batch_id", "_silver_load_timestamp", "_silver_batch_id"]
has_lineage = all(c in verify_clean.columns for c in required_lineage)
print(f"  Lineage cols:  {'present' if has_lineage else 'MISSING'}")
print()

print("Reject breakdown (from rejects table):")
verify_rejects.groupBy("reject_reason").count().orderBy("count", ascending=False).show(truncate=False)

print("Sample clean rows:")
verify_clean.select(
    "policy_id", "program_code", "carrier_id", "agent_id",
    "premium", "status", "_source_system", "_silver_batch_id"
).show(5, truncate=False)

StatementMeta(, 7ee4c904-f779-4dea-a6d9-2234662fe186, 11, Finished, Available, Finished, False)

silver_policies verified:
  Clean rows:    9709
  Rejected rows: 291
  Total deduped: 10000
  Math checks:   10000 == 10000: True

  Lineage cols:  present

Reject breakdown (from rejects table):
+------------------------+-----+
|reject_reason           |count|
+------------------------+-----+
|null premium            |80   |
|negative premium        |62   |
|invalid agent_id        |57   |
|malformed effective_date|49   |
|invalid carrier_id      |43   |
+------------------------+-----+

Sample clean rows:
+----------+------------+----------+--------+--------+---------+--------------+-------------------------------+
|policy_id |program_code|carrier_id|agent_id|premium |status   |_source_system|_silver_batch_id               |
+----------+------------+----------+--------+--------+---------+--------------+-------------------------------+
|POL-000001|ENRG        |CAR-018   |AGT-108 |30000.0 |ACTIVE   |csv           |silver_20260603_010842_2e7daf15|
|POL-000003|CONT        |CAR-009   |AGT